# Lab 3 - Multi-Provider LLM APIs
**COM SCI-X 455 - Week 3 - Megan Armani, PhD**

> Push to `genai-course/week3/`

Keys come from Colab Secrets only (`UCLA_OPENAI_API_KEY` or `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, optional `HF_TOKEN` for Llama 3).

## Setup

In [ ]:
!pip install -q -U openai anthropic
!pip install -q rouge-score tabulate

In [ ]:
import os
def get_secret(*names):
    """Read a key from Colab Secrets (or env vars). Keys are never printed."""
    try:
        from google.colab import userdata
        for n in names:
            try:
                v = userdata.get(n)
                if v:
                    return v
            except Exception:
                pass
    except ImportError:
        pass
    for n in names:
        if os.environ.get(n):
            return os.environ[n]
    return None

OPENAI_KEY = get_secret("UCLA_OPENAI_API_KEY", "OPENAI_API_KEY")
ANTHROPIC_KEY = get_secret("ANTHROPIC_API_KEY", "UCLA_ANTHROPIC_API_KEY", "CLAUDE_API_KEY")
if OPENAI_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_KEY
if ANTHROPIC_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY
print("OpenAI key found:", bool(OPENAI_KEY), "| Anthropic key found:", bool(ANTHROPIC_KEY))

## Part 1 - Basic API Calls: OpenAI + Anthropic + Llama 3

In [ ]:
import time, json, random, re
from openai import OpenAI
import anthropic

# USD per 1M tokens (input, output). Official list prices; check the provider pages before reuse.
PRICES = {"gpt-4o": (2.50, 10.00), "gpt-4o-mini": (0.15, 0.60),
          "claude-sonnet-4-5": (3.00, 15.00), "llama-3.1-8b (HF)": (0.0, 0.0)}
GPT_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-sonnet-4-5"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

oa = OpenAI() if OPENAI_KEY else None
an = anthropic.Anthropic() if ANTHROPIC_KEY else None
HF_TOKEN = get_secret("HF_TOKEN", "HUGGINGFACE_TOKEN")
hf = None
if HF_TOKEN:
    try:
        from huggingface_hub import InferenceClient
        hf = InferenceClient(model=LLAMA_MODEL, token=HF_TOKEN)
    except Exception as e:
        print("Llama (HF) not available:", e)

def with_retry(fn, tries=4):
    """Call fn(); on API errors wait 1s, 2s, 4s (+ jitter) and try again."""
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            if i == tries - 1:
                raise
            print(f"  retry {i+1}: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(2 ** i + random.random())

def call(model, user, system=None, temperature=0, max_tokens=1200, json_mode=False):
    """One chat call to any provider. Returns text, latency, tokens and cost."""
    t0 = time.time()
    if model.startswith("gpt"):
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
        kw = dict(model=model, messages=msgs, temperature=temperature, max_tokens=max_tokens)
        if json_mode:
            kw["response_format"] = {"type": "json_object"}
        r = with_retry(lambda: oa.chat.completions.create(**kw))
        text, tin, tout = r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens
    elif model.startswith("claude"):
        kw = dict(model=model, max_tokens=max_tokens, temperature=temperature,
                  messages=[{"role": "user", "content": user}])
        if system:
            kw["system"] = system          # Anthropic: system is a top-level parameter
        r = with_retry(lambda: an.messages.create(**kw))
        text = "".join(b.text for b in r.content if b.type == "text")
        tin, tout = r.usage.input_tokens, r.usage.output_tokens
    else:  # Llama 3 via Hugging Face Inference API
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
        r = with_retry(lambda: hf.chat_completion(messages=msgs, max_tokens=max_tokens, temperature=max(temperature, 0.01)))
        text = r.choices[0].message.content
        tin = getattr(r.usage, "prompt_tokens", 0) if r.usage else 0
        tout = getattr(r.usage, "completion_tokens", 0) if r.usage else 0
    lat = time.time() - t0
    pin, pout = PRICES.get(model, (0, 0))
    return {"model": model, "text": text, "latency_s": round(lat, 2), "in_tok": tin, "out_tok": tout,
            "cost_usd": round(tin / 1e6 * pin + tout / 1e6 * pout, 6)}

MODELS = []
if oa: MODELS.append(GPT_MODEL)
if an: MODELS.append(CLAUDE_MODEL)
if hf: MODELS.append("llama-3.1-8b (HF)")
print("Models in this run:", MODELS)
if len(MODELS) < 2:
    print("WARNING: add ANTHROPIC_API_KEY (or HF_TOKEN) to Colab Secrets to compare two providers.")

def parse_json(text):
    """Pull the first JSON object out of a model reply (handles ```json fences)."""
    m = re.search(r"\{.*\}", text, re.S)
    return json.loads(m.group(0)) if m else None

ARTICLE = """
The James Webb Space Telescope has captured the deepest infrared image of the universe ever taken.
The image shows thousands of galaxies, some dating back over 13 billion years.
Scientists believe this will revolutionise our understanding of early galaxy formation.
"""
SUMMARY_PROMPT = f"Summarise this article in exactly 2 sentences.\n\n{ARTICLE}"

In [ ]:
# GPT-4o
gpt = call(GPT_MODEL, SUMMARY_PROMPT, system="You are a concise science editor.")
print(gpt["text"]); print({k: gpt[k] for k in ("latency_s", "in_tok", "out_tok", "cost_usd")})

In [ ]:
# Claude: system= is a top-level parameter in the Anthropic API, not a message
claude = call(CLAUDE_MODEL, SUMMARY_PROMPT, system="You are a concise science editor.") if an else None
print(claude["text"] if claude else "Claude skipped: no ANTHROPIC_API_KEY in Secrets")

In [ ]:
# Llama 3 via Hugging Face Inference API (optional: needs HF_TOKEN with Llama access)
llama = None
if hf:
    try:
        llama = call("llama-3.1-8b (HF)", SUMMARY_PROMPT, system="You are a concise science editor.")
    except Exception as e:
        print("Llama call failed:", e)
summaries = {"GPT-4o": gpt, "Claude": claude, "Llama 3": llama}
for name, r in summaries.items():
    print(f"--- {name} ---")
    print(r["text"] if r else "(not run)")

## Part 2 - Streaming

In [ ]:
LONG_PROMPT = "Explain in about 150 words why infrared lets JWST see very old galaxies."

t0 = time.time(); first = None; parts = []
stream = oa.chat.completions.create(model=GPT_MODEL, messages=[{"role": "user", "content": LONG_PROMPT}],
                                    temperature=0, stream=True)
for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        if first is None:
            first = time.time() - t0
        parts.append(delta); print(delta, end="", flush=True)
stream_total = time.time() - t0

t0 = time.time()
oa.chat.completions.create(model=GPT_MODEL, messages=[{"role": "user", "content": LONG_PROMPT}], temperature=0)
nonstream_total = time.time() - t0
print(f"\n\nStreaming: time to first token {first:.2f}s, total {stream_total:.2f}s")
print(f"Non-streaming: user waits {nonstream_total:.2f}s before seeing anything")

## Part 3 - Structured Output with Pydantic

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal

class ArticleSummary(BaseModel):
    title:      str
    summary:    str   # exactly 2 sentences
    sentiment:  Literal['positive', 'negative', 'neutral']
    confidence: float # 0.0 to 1.0
    key_claims: list[str]  # max 3 items

    @field_validator('confidence')
    def confidence_range(cls, v):
        if not 0 <= v <= 1:
            raise ValueError('confidence must be 0-1')
        return v

    @field_validator('key_claims')
    def max_three(cls, v):
        if len(v) > 3:
            raise ValueError('max 3 key claims')
        return v

In [ ]:
schema = json.dumps(ArticleSummary.model_json_schema()["properties"], indent=1)
r = oa.chat.completions.create(
    model=GPT_MODEL, temperature=0, response_format={"type": "json_object"},
    messages=[{"role": "system", "content": "Return only JSON with keys title, summary, sentiment, confidence, key_claims."},
              {"role": "user", "content": f"Schema:\n{schema}\n\nArticle:\n{ARTICLE}"}])
raw = r.choices[0].message.content
parsed = ArticleSummary.model_validate_json(raw)
print("Valid:", parsed.model_dump())

# Failure case: confidence = 1.5 must be rejected by the validator
bad = json.loads(raw); bad["confidence"] = 1.5
try:
    ArticleSummary.model_validate(bad)
    print("Validator did NOT catch it")
except ValidationError as e:
    print("Validator caught it:", e.errors()[0]["msg"])

# Second failure case: 4 key claims
bad = json.loads(raw); bad["key_claims"] = ["a", "b", "c", "d"]
try:
    ArticleSummary.model_validate(bad)
except ValidationError as e:
    print("Validator caught it:", e.errors()[0]["msg"])

## Part 4 - Evaluation with ROUGE

In [ ]:
from rouge_score import rouge_scorer
import pandas as pd
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

REFERENCE = "The James Webb Space Telescope captured the universe's deepest infrared image, showing thousands of ancient galaxies that will transform our knowledge of early cosmic history."

rows = []
for name, r in summaries.items():
    if not r:
        continue
    s = scorer.score(REFERENCE, r["text"])
    rows.append({"Model": name, "ROUGE-1": round(s["rouge1"].fmeasure, 3),
                 "ROUGE-2": round(s["rouge2"].fmeasure, 3), "ROUGE-L": round(s["rougeL"].fmeasure, 3),
                 "Words": len(r["text"].split())})
rouge_df = pd.DataFrame(rows).sort_values("ROUGE-L", ascending=False)
print(rouge_df.to_markdown(index=False))
print("\nHighest ROUGE-L:", rouge_df.iloc[0]["Model"])

### Reflect
1. Did GPT-4o, Claude, and Llama 3 produce meaningfully different summaries? Describe the differences.
2. Your ROUGE scores are based on one reference. What's wrong with this evaluation methodology?
3. What would a better evaluation framework look like for summarisation quality?

### Reflection

**1. Differences.** The models agree on the facts (deepest infrared image, galaxies over 13 billion years old,
early galaxy formation), because the article is only three sentences long. The differences are in style:
length, whether "revolutionise" is kept or softened, and whether the second sentence adds framing that is not
in the article. The printed outputs and word counts above show which model stayed closest to the source wording.

**2. Problem with one reference.** ROUGE counts overlapping words and n-grams against a single human summary.
A correct paraphrase that uses different words ("oldest" instead of "ancient") is punished, and a summary that
copies the reference wording but adds a wrong claim is rewarded. One reference also hides the many valid ways
to summarise the same text, so small ROUGE differences between models are mostly noise.

**3. Better framework.** Use several references, add an embedding or BERTScore metric for meaning, and check
faithfulness separately (every claim in the summary must be supported by the article, checked by an NLI model or
an LLM judge with a rubric). Add format checks (exactly 2 sentences) and a small human preference test on a
larger set of articles, not one.

---
## Submission
- [x] Push to `genai-course/week3/lab3.ipynb`
- [x] All 3 models called and compared (Llama 3 runs when `HF_TOKEN` is set)
- [x] Pydantic validation demonstrated (including a failure case)
- [x] ROUGE table completed

## Part 5 - Hallucination Detection
**Goal:** Identify hallucinated claims in model output. This feeds directly into the homework hallucination audit.

In [ ]:
HALLU_PROMPTS = [
    {"id": "statistic", "prompt": "What was OpenAI's exact revenue in Q3 2019, in US dollars?",
     "truth": "OpenAI did not publish quarterly revenue for Q3 2019. Any exact number is unverifiable or made up."},
    {"id": "fictional_event", "prompt": "Summarise the main provisions of the 2019 Reykjavik Treaty on Artificial Intelligence Safety signed by the Nordic countries.",
     "truth": "No such treaty exists. A correct answer says it cannot find or does not know this treaty."},
    {"id": "citation", "prompt": "Give the full citation (authors, title, venue, year, volume, pages and DOI) of the 2012 paper that introduced AlexNet.",
     "truth": "Krizhevsky, Sutskever, Hinton. ImageNet Classification with Deep Convolutional Neural Networks. NIPS 2012 (Advances in Neural Information Processing Systems 25, pp. 1097-1105). The NIPS paper has no DOI; the 2017 Communications of the ACM reprint has DOI 10.1145/3065386."},
]

def judge_claims(answer, truth):
    """GPT-4o judge compares one answer with my ground-truth note. I spot-check every verdict."""
    out = call(GPT_MODEL, f"""Ground truth note: {truth}

Model answer: {answer}

List the main factual claim in the answer and give a verdict.
CORRECT = consistent with the ground truth, HALLUCINATED = states something false or invented as fact,
UNVERIFIABLE = cannot be checked against the ground truth.
Return JSON: {{"claim": str, "verdict": "CORRECT|HALLUCINATED|UNVERIFIABLE"}}""", json_mode=True, max_tokens=300)
    return json.loads(out["text"])

hallu_rows = []
for p in HALLU_PROMPTS:
    for m in [x for x in MODELS if not x.startswith("llama")]:
        ans = call(m, p["prompt"], temperature=0, max_tokens=400)["text"]
        v = judge_claims(ans, p["truth"])
        hallu_rows.append({"Model": m, "Prompt": p["id"], "Claim": v["claim"][:110], "Verdict": v["verdict"]})
        print(f"[{m} | {p['id']}] {ans[:300]}\n  -> {v['verdict']}\n")
hallu_df = pd.DataFrame(hallu_rows)
print(hallu_df.to_markdown(index=False))

### Reflect - Hallucination
1. Which model hallucinated more? Was it confident when wrong?
2. What mitigation (RAG, temperature=0, structured output) would help most for your specific prompts?
3. 'The model made it up' is not an explanation. What IS the architectural reason hallucination occurs?

### Reflection

**1. Which model hallucinated more.** See the verdict table above. The citation prompt is the most dangerous one:
author names and title are usually right, but extra fields (DOI, page numbers) are where invented details appear,
and they are written in the same confident tone as the correct fields. The fictional-treaty prompt tests whether a
model refuses a false premise or invents provisions.

**2. Best mitigation.** For the statistic and citation prompts, retrieval (RAG over a trusted source such as a
paper database) helps most, because the model only needs to copy fields that exist. For the false-premise prompt,
an explicit instruction ("if you cannot verify it, say you do not know") plus a structured field like
`verified: bool` helps more. Temperature 0 makes answers repeatable but does not make them true.

**3. Architectural reason.** An LLM is trained to predict the most likely next token, not to look up facts.
There is no separate store of verified facts and no built-in signal for "I have not seen this". When a prompt
asks for a specific number or DOI, the most likely continuation is something shaped like a number or DOI, so
the model produces a fluent, plausible string. Rare facts are stored weakly in the weights, and training rewards
complete answers more than abstaining, so the model fills gaps instead of refusing.